# Практика 1. От workflow к агенту на LangGraph

Ты построишь **две архитектуры для одной задачи**: граф с заданным порядком действий и агентный цикл, в котором следующий инструмент выбирает модель.

**Что пишешь ты:** фильтр предложений, граф workflow, исполнитель инструментов и граф агента. **Что уже готово:** маленький каталог, инструменты чтения, учебные ответы модели и проверки.

В каждой ячейке с `TODO` замени `raise NotImplementedError` своим кодом. Проверки сразу ниже запускай после решения соответствующего задания. До заполнения всех заданий `Run All` закономерно остановится на первой заглушке. Полного решения в тетради нет.

Основная часть работает **локально без LLM, ключа и скачивания весов**. LangGraph настоящий; решения модели в проверках заранее записаны. В конце есть отдельный запуск с настоящей моделью через Hugging Face. Он отключён по умолчанию.

Это одна работа на несколько подходов. Цель — объяснить исполнение по журналу и уметь изменить граф, а не только получить зелёные проверки.

## 0. Окружение и карта задачи

1. Выполни команды из соседнего `README.md` и выбери ядро созданного окружения в Jupyter/VS Code.
2. Прочитай в конспекте первый раздел про агентный цикл. Остальной текст главы ещё будет перерабатываться; здесь все контракты заданы явно.
3. Запусти готовые ячейки ниже.

Весь код работает на небольших Python-словарях. Не нужны веб-сервер, база, GPU или Langfuse. Если уже установлен другой LangGraph, используй отдельное окружение: версии закреплены в `requirements.txt`.

In [ ]:
from pathlib import Path
import sys

here = Path.cwd()
candidates = [here, *[p / "practice" / "agents" for p in (here, *here.parents)]]
LAB_DIR = next((p for p in candidates if (p / "lab01_support.py").exists()), None)
if LAB_DIR is None:
    raise FileNotFoundError("Открой тетрадь из репозитория вместе с lab01_support.py")
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

import json
from copy import deepcopy
from datetime import date
from importlib.metadata import version
from pydantic import ValidationError
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langgraph.graph import StateGraph, START, END
from lab01_support import (
    OFFERS, CONSTRAINTS, TOOLS, TOOL_REGISTRY, SYSTEM,
    WorkflowState, AgentState, ScriptedModel, LoopingModel, HFModel,
    workflow_input, agent_input, call_message, happy_script, print_history,
)
from lab01_checks import check_filter, check_workflow, check_executor, check_agent

print({p: version(p) for p in ("langgraph", "langchain-core", "huggingface-hub")})

**Учебный сценарий:** разговор 8 октября 2026 года. Нужен бумажный «Маленький принц» на русском до 1500 рублей **с доставкой**, получить не позднее 9 октября. Дата фиксирована и не зависит от сегодняшнего дня.

| Предложение | Язык | Книга | Доставка | Получение |
|---|---|---:|---:|---|
| A, «Лист» | ru | 950 | 350 | 2026-10-09 |
| B, «Полка» | ru | 900 | 650 | 2026-10-09 |
| C, «Том» | en | 800 | 200 | 2026-10-08 |

Магазины и данные вымышлены. `search_books` возвращает предложения **без доставки**. `get_offer_details` читает полные условия по `id`. Эти два инструмента ничего не покупают.

Название Python-функции и её описание превращаются в схему инструмента. При удалённом вызове модель увидит схему, а выполнять функцию будет твой код.

In [ ]:
print("Условия:", CONSTRAINTS)
print("Поиск:", search_result := TOOL_REGISTRY["search_books"].invoke(
    {"title": "Маленький принц"}
))
print("Полные условия A:", TOOL_REGISTRY["get_offer_details"].invoke({"offer_id": "A"}))
print("Схема аргументов:", TOOL_REGISTRY["get_offer_details"].args_schema.model_json_schema())

### Минимум LangGraph перед заданием

`StateGraph(тип_состояния)` задаёт общую структуру данных. Узел — Python-функция: получает текущее состояние и возвращает **обновление отдельных полей**. `add_edge` задаёт следующий узел; `add_conditional_edges` выбирает его по результату функции маршрутизации. `compile()` собирает исполняемый граф, `invoke()` запускает его.

В `WorkflowState` обычные поля заменяются новым значением. В `AgentState.messages` подключено правило `add_messages`: новые сообщения добавляются, а сообщение с уже существующим `id` обновляется. Поэтому узел модели возвращает только новое сообщение, а не копию всей истории.

Ниже короткий готовый пример обновления состояния. Он показывает синтаксис, но не решает задания.

In [ ]:
from typing import TypedDict

class CounterState(TypedDict):
    value: int

def increment(state):
    return {"value": state["value"] + 1}

example = StateGraph(CounterState)
example.add_node("increment", increment)
example.add_edge(START, "increment")
example.add_edge("increment", END)
print(example.compile().invoke({"value": 2}))  # {'value': 3}

## 1. Проверяем условия задачи обычным кодом

Реализуй `eligible_offers(offers, constraints)`. На входе уже полные условия предложений. Верни подходящие словари, не изменяя входные объекты.

Все ограничения обязательны: совпадают название, язык и формат; сумма книги и доставки не превышает `max_total`; дата `arrives_on` не позднее `deadline`. Даты можно преобразовать через `date.fromisoformat`. В этом наборе цена и доставка всегда известны.

**До запуска:** почему B не подходит при бюджете 1500, хотя сама книга дешевле A? Что изменится при бюджете 1550?

In [ ]:
def eligible_offers(offers, constraints):
    # TODO: верни список подходящих предложений.
    raise NotImplementedError("Реализуй проверку условий")

In [ ]:
check_filter(eligible_offers)

## 2. Собираем workflow с фиксированным порядком

Реализуй `build_workflow()`, возвращающую скомпилированный `StateGraph(WorkflowState)`.

Нужны узлы:

- `search`: вызвать `search_books` с названием из `state["constraints"]`, записать предложения в `offers`;
- `details`: прочитать полные условия найденных id через `get_offer_details`, заменить `offers`;
- `select`: записать результат твоей `eligible_offers` в `eligible`;
- `answer`: сформировать непустой текст и установить `status="completed"`;
- `no_match`: сформировать непустой текст и установить `status="no_match"`.

Путь: `START → search → details → select`. После `select` сделай условный переход: есть варианты — `answer`, иначе — `no_match`. Обе ветви завершаются в `END`. Пустой поиск должен корректно пройти через граф.

Формулировку ответа выбираешь сам. Этот workflow не использует модель: порядок действий задан тобой.

<details><summary>Подсказка по синтаксису</summary>

Добавить узел: `builder.add_node("search", функция)`. Маршрут: `builder.add_conditional_edges("select", route, {"yes": "answer", "no": "no_match"})`. Функция `route(state)` возвращает один из ключей словаря. Не запускай отдельный `invoke()` внутри каждого узла.
</details>

In [ ]:
def build_workflow():
    # TODO: определи функции узлов, собери переходы и верни builder.compile().
    raise NotImplementedError("Собери workflow")

In [ ]:
check_workflow(build_workflow)
workflow = build_workflow()
for update in workflow.stream(workflow_input(), stream_mode="updates"):
    print(update)

**Запиши наблюдение:** какие действия workflow выполняет всегда? Какие из них могли бы быть лишними, если пользователь уже дал id конкретного предложения?

In [ ]:
workflow_observation = ""  # Напиши 2–3 предложения.

## 3. Исполняем решения модели

Модель возвращает `AIMessage`. В нём может быть обычный `content` либо список `tool_calls`. Один вызов имеет поля `name`, `args` и `id`. Это **предложение выполнить функцию**.

Реализуй `execute_calls(message, registry) → list[ToolMessage]`:

1. Обработай каждый вызов в исходном порядке.
2. Если имени нет в `registry`, верни результат с `ok=False`, `error="unknown_tool"`.
3. Проверь аргументы через `selected_tool.args_schema.model_validate(call["args"])`. `ValidationError` означает `error="invalid_arguments"`.
4. Исполни `selected_tool.invoke(call["args"])`. Успех: `{"ok": True, "data": результат}`. Ожидаемую ошибку функции `ValueError`/`OSError` верни как `error="tool_error"` с коротким `details`. Не перехватывай все исключения: ошибку своего кода нужно увидеть.
5. Для каждого вызова создай `ToolMessage(content=json.dumps(payload, ensure_ascii=False), tool_call_id=call["id"])`.

Пустой список вызовов даёт пустой список результатов. Исходное сообщение не меняй. Разделяй проверку аргументов и исполнение: `ValidationError` является подклассом `ValueError`.

Несколько вызовов в одном сообщении здесь **исполняются последовательно**. Мы изучаем формат пакета; параллелизм в этой работе не требуется.

In [ ]:
def execute_calls(message, registry=TOOL_REGISTRY):
    # TODO: проверка имени, аргументов, исполнение и ToolMessage для каждого id.
    raise NotImplementedError("Реализуй исполнителя")

In [ ]:
check_executor(execute_calls)

## 4. Замыкаем агентный цикл

Теперь порядок вызовов приходит от объекта `model`. Для локальных проверок `ScriptedModel` возвращает записанную последовательность `AIMessage`. Это **не LLM и не проверка качества выбора**: она проверяет, что граф правильно передаёт решения и наблюдения. Тот же интерфейс будет у `HFModel`.

Реализуй `build_agent(model, max_steps=6)`. Верни скомпилированный `StateGraph(AgentState)` с узлами `model`, `tools`, `limit`.

**Контракт узлов:**

- `model`: вызвать `model.invoke([SystemMessage(content=SYSTEM), *state["messages"]])`; вернуть обновление с новым `AIMessage` и увеличить `steps` на один. При обычном текстовом ответе также записать `answer` и `status="completed"`. Пустой ответ без вызовов считай ошибкой (`ValueError`).
- `tools`: вернуть новые сообщения из `execute_calls` для последнего сообщения модели. Счётчик решений здесь не меняется.
- `limit`: установить `status="step_limit"` и непустое объяснение в `answer`, не обращаясь к модели.

**Переходы:**

- `START → model`;
- после модели: есть `tool_calls` — в `tools`, иначе — в `END`;
- после инструментов: если `steps < max_steps`, обратно в `model`, иначе — в `limit`;
- `limit → END`.

Один шаг бюджета — одно решение модели, включая финальный ответ. Если последнее разрешённое решение предложило инструменты, исполни их и сохрани ответы, затем остановись. Так в журнале не останется вызовов без результатов. Например, `max_steps=2` недостаточно для сценария «поиск → три проверки одним ходом → ответ»: ему нужны три решения.

При построении графа отклоняй `max_steps`, который не является положительным `int` (в том числе `True`). Лимит шагов — часть программы; инструкция в промпте его не заменяет.

<details><summary>На что смотреть, если теряется история</summary>

Узел возвращает `{"messages": [новое_сообщение]}`, а правило поля `add_messages` объединяет обновление со старым состоянием. Не возвращай только последнее наблюдение из пакета. Каждое `ToolMessage` должно сохранить id соответствующего вызова.
</details>

In [ ]:
def build_agent(model, max_steps=6):
    # TODO: узлы, маршрутизация, проверка бюджета и compile().
    raise NotImplementedError("Собери агентный цикл")

In [ ]:
check_agent(build_agent)
model = ScriptedModel(happy_script())
agent = build_agent(model, max_steps=5)
result = agent.invoke(agent_input())
print("Статус:", result["status"], "Решений модели:", result["steps"])
print_history(result["messages"])

### Самостоятельный сценарий

Проверки уже покрывают исправление id предложения и бесконечный поиск. Добавь **свой сценарий**: первая попытка вызывает неизвестный инструмент, следующая использует допустимый инструмент, затем модель отвечает. Перед запуском запиши ожидаемые `status`, `steps` и число сообщений инструментов. Проверь их через `assert`.

Для создания одного решения используй `call_message(("имя", {"аргумент": "значение"}, "уникальный-id"))`. Для финального ответа — `AIMessage(content="текст")`.

In [ ]:
# TODO: predictions, ScriptedModel([...]), запуск и проверки.
# Используй build_agent; не вызывай инструменты вручную в обход графа.

## 5. Подключение модели через Hugging Face

Этот этап выполняй после локальных проверок. Он меняет источник решений, сохраняя твой граф и инструменты.

Hugging Face Hub хранит репозитории моделей. **Inference Providers** даёт удалённое выполнение поддерживаемых моделей у провайдеров. Наличие модели на Hub ещё не означает наличие совместимого сервера. Нам нужны chat completion и function/tool calling у выбранной пары «модель — провайдер».

1. Создай аккаунт HF и [fine-grained токен](https://huggingface.co/settings/tokens) с правом **Make calls to Inference Providers**. Сам токен в чат и в ячейку не вставляй: ниже он вводится скрыто через `getpass`.
2. Проверь [условия оплаты](https://huggingface.co/docs/inference-providers/pricing). На 2026-10-10 бесплатному аккаунту нужны купленные кредиты; не рассчитывай на бесплатную квоту только за регистрацию. Локальная часть работы оплаты не требует.
3. В [каталоге Inference Providers](https://huggingface.co/inference/models) выбери доступную чат-модель с поддержкой tools. Заполни `HF_MODEL` и при необходимости `HF_PROVIDER`. Поддержку tools проверь по документации провайдера; доступность модели и баланс аккаунта здесь не проверялись.
4. Для своего ручного прогона поставь `RUN_REMOTE=True` и выполни следующую ячейку. Это отправит учебную задачу, описания функций и результаты инструментов выбранному провайдеру и расходует кредиты. Веса на компьютер не скачиваются.

`HFModel` — готовый небольшой адаптер в `lab01_support.py`: переводит сообщения LangChain в формат HF, посылает схемы инструментов и возвращает `AIMessage`. Его можно прочитать после задания. Сам адаптер инструменты не исполняет.

Ошибки доступа/баланса и несовместимость tools разбирай отдельно от ошибок графа. При обрыве ответа по лимиту токенов адаптер выдаст ошибку; это не успешное завершение агента. Для модели с длительным рассуждением заданного в адаптере резерва 1024 может быть мало.

In [ ]:
RUN_REMOTE = False
HF_MODEL = ""         # ID выбранной обслуживаемой модели, например организация/имя.
HF_PROVIDER = "auto"  # Либо конкретный провайдер, поддерживающий эту модель и tools.

if RUN_REMOTE:
    from getpass import getpass
    if not HF_MODEL.strip():
        raise ValueError("Сначала заполни HF_MODEL")
    hf_model = HFModel(
        model_id=HF_MODEL, provider=HF_PROVIDER,
        token=getpass("HF token (ввод скрыт): "),
    )
    live_agent = build_agent(hf_model, max_steps=6)
    live_result = live_agent.invoke(agent_input(), config={"recursion_limit": 30})
    print(live_result["status"], live_result["answer"])
    print_history(live_result["messages"])
    print("Расход по ответам провайдера:", hf_model.usage)
else:
    print("Удалённая модель выключена. Все предыдущие задания доступны без ключа.")

**После ручного запуска:** сравни путь модели с workflow. Она проверила доставку? Прочитала лишние предложения? Взяла цену из наблюдения? Уложилась в бюджет решений? Прохождение локальных тестов этих свойств не гарантирует.

Для второго небольшого эксперимента можно снизить бюджет до 1250 через `agent_input(max_total=1250)`. Запускай вручную: у этой задачи нет подходящего варианта. Корректный отказ здесь лучше выдуманного предложения.

In [ ]:
experiment_notes = ""  # Что наблюдал: последовательность, итог, одна ошибка или лишнее действие.

## 6. Что прислать на разбор

Сохрани тетрадь со своим кодом и ответами. Пришли путь к файлу и место, где возник вопрос; мы разберём его прежде, чем я покажу решение.

Работа готова, когда:

- четыре группы локальных проверок проходят после перезапуска ядра;
- есть твой сценарий восстановления после неизвестного инструмента;
- ты можешь по журналу объяснить различие решения модели и исполнения инструмента;
- ты объясняешь, зачем нужны `add_messages`, `tool_call_id` и программный предел решений;
- можешь показать, где заменить учебную модель настоящей. Удалённый эксперимент можно отложить до появления доступа.

**Вопросы для короткого ответа:** почему `bind_tools` или передача `tools` в запрос не являются обучением? Сколько решений и сколько исполнений инструментов в `happy_script`? Что произойдёт при возвращении из узла всей старой истории с новыми id? Где в твоём коде выбирается следующий инструмент, а где — следующий узел графа?

В третьей большой практике добавим **Langfuse**: инструмент для просмотра трасс запросов, задержек, расхода токенов и результатов оценок. LangGraph управляет исполнением; Langfuse помогает исследовать получившиеся запуски. Здесь достаточно `stream()` и текстового журнала. Подключение Langfuse будет отдельным явным действием, поскольку оно передаёт трассы настроенному серверу.

Источники: [LangGraph Quickstart](https://docs.langchain.com/oss/python/langgraph/quickstart), [Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api), [тестирование](https://docs.langchain.com/oss/python/langgraph/test), [HF function calling](https://huggingface.co/docs/inference-providers/guides/function-calling), [HF InferenceClient](https://huggingface.co/docs/huggingface_hub/package_reference/inference_client), [интеграция Langfuse](https://langfuse.com/integrations/frameworks/langchain).

Границы проверки автором: служебный код и задания проверены с временной контрольной реализацией на настоящем LangGraph; адаптер HF — с подставным ответом клиента. Настоящая модель не вызывалась. В поставляемой тетради решения и результаты запусков отсутствуют намеренно.

In [ ]:
answers = {
    "tools_vs_training": "",
    "decisions_vs_executions": "",
    "history_updates": "",
    "two_kinds_of_choice": "",
}